# Exploratory Data Analysis (EDA) - `glassdoor_reviews.csv`

This notebook performs an Exploratory Data Analysis (EDA) on the `glassdoor_reviews.csv` dataset. The goal is to understand the dataset's structure, identify potential issues like missing values or duplicates, and derive key insights into the distribution of the target variable and feature correlations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline

## 1. Load Data
Loading the `glassdoor_reviews.csv` dataset.

In [ ]:
data_path = Path('../src/data/glassdoor_reviews.csv')
df = pd.read_csv(data_path)
print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")

## 2. Data Overview
Inspecting the first few rows, data types, and basic statistics.

In [ ]:
display(df.head())
display(df.info())
display(df.describe())

## 3. Missing Values Analysis
Identifying and visualizing missing data.

In [ ]:
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_data, 'Missing Percent (%)': missing_percent})
display(missing_df)

plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Data Heatmap')
plt.show()

## 4. Duplicate Rows Analysis
Checking for and quantifying duplicate entries.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    print("Consider removing duplicate rows if they represent identical entries, not just similar ones.")

## 5. Target Variable Distribution (`overall_rating`)
Visualizing the distribution of the `overall_rating`.

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='overall_rating', data=df, palette='viridis')
plt.title('Distribution of Overall Rating')
plt.xlabel('Overall Rating')
plt.ylabel('Count')
plt.show()

## 6. Feature Correlation Analysis
Analyzing correlations between numerical features, especially with the target variable.

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
if 'overall_rating' in numerical_cols:
    correlation_matrix = df[numerical_cols].corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Matrix of Numerical Features')
    plt.show()

    print("Correlations with Overall Rating:")
    print(correlation_matrix['overall_rating'].sort_values(ascending=False))

## 7. Key Insights

Based on the EDA, here are some initial insights:

1.  **Missing Data:** There are significant missing values in columns like `work_life_balance`, `senior_mgmt`, and `diversity_inclusion`. The `clean_data` function in `src/data/make_dataset.py` handles these, primarily through imputation or by calculating a 'belonging score' from available pillars.
2.  **Target Distribution:** The `overall_rating` is heavily skewed towards higher ratings (4s and 5s), which is common for review data. This might impact model performance for lower ratings and suggests considering evaluation metrics beyond simple RMSE/MAE, or techniques like stratified sampling.
3.  **Feature Importance (Initial):** `culture_values` and `career_opp` show the strongest positive correlation with `overall_rating`. This aligns with the project's focus on Purpose and Growth.
4.  **Categorical Data:** Columns like 'firm', 'job_title', and 'location' are highly cardinal. Effective encoding strategies (e.g., target encoding, embedding) will be crucial for these features if used directly in modeling.
5.  **Data Cleaning:** The preprocessing logic in `make_dataset.py` correctly addresses several issues identified in this EDA, such as handling missing values and creating composite scores. However, further exploration into the impact of these imputations could be beneficial.